# Step 9: self-correction loop on the fine-tuned model (Run A final, prompt v2)

Generates SQL for every question, executes it on the real SQLite database, and when the
query fails, feeds that back to the model and asks again (at most 2 retries). The loop
only uses signals available at inference time: the SQLite error message, or an empty
result. No gold SQL is uploaded to Kaggle at all.

One session measures every variant on the same first attempt:

| output file | retries on | what the model is told |
|---|---|---|
| `predictions_attempt1.jsonl` | nothing (no-retry baseline) | - |
| `selfcorrect_plain.jsonl` | execution error | only that the query was wrong |
| `selfcorrect_error.jsonl` | execution error | the SQLite error message |
| `selfcorrect_plain_empty.jsonl` | error or empty result | only that the query was wrong |
| `selfcorrect_error_empty.jsonl` | error or empty result | the error message, or that no rows came back |

Decoding is the same as every earlier run: greedy, `max_new_tokens=256`, stop at
`<|im_end|>`, fp16, adapter merged into the base model.

**Before running:**
- Settings -> Accelerator: **GPU T4 x2** (one T4 is used), Internet: **On**.
- Add Input -> the prompts dataset (needs `prompts_v2_dev.jsonl`).
- Add Input -> the databases dataset (the `.sqlite` files written by
  `scripts/prepare_kaggle_databases.py`).
- Add-ons -> Secrets: **`HF_TOKEN`** (read access is enough).
- **Save Version -> Save & Run All**, not an interactive Run All. Only a saved Version
  keeps its output files.

Expected run time on a T4: about 8 minutes for the first attempt, then a few minutes of
retries (under 200 questions per round). Around 20-30 minutes in total.

## How to resume an interrupted run
1. The interrupted run must have been started with **Save Version -> Save & Run All**.
2. Open this notebook again. **Add Input** -> **Notebook Output Files** tab -> this
   notebook -> pick the version to resume from -> **Add**. Keep both datasets attached.
3. **Save Version -> Save & Run All** again. The notebook copies
   `predictions_attempt1.jsonl` and `retry_generation_cache.jsonl` from the attached
   output into `/kaggle/working`, continues the first attempt from the first missing row,
   and reuses every retry that was already generated. The `selfcorrect_*.jsonl` files are
   rebuilt from those two files, so nothing is generated twice.


In [ ]:
# Environment check -- catch a broken environment before spending any GPU time.
import json as _json
import os
import subprocess
import sys

print("Python:", sys.version)

import torch
print("torch:", torch.__version__, "| torch's CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU available in this session. Check Settings -> Accelerator is set to GPU T4 x2, "
        "then Run All again."
    )

print("GPU:", torch.cuda.get_device_name(0))

import transformers
print("transformers:", transformers.__version__)

# Packages a pip install must never be allowed to touch -- this is exactly what broke an
# earlier notebook's vLLM install (pip silently upgraded torchaudio to a build that didn't
# match the already-loaded torch).
WATCHED_PACKAGES = ["torch", "torchvision", "torchaudio", "triton", "transformers"]


def installed_torch_build():
    """What's actually on disk right now, read via a FRESH subprocess -- not the
    already-imported `torch` module in this kernel, which can't reflect a pip install
    that happened after it was first imported."""
    r = subprocess.run(
        [sys.executable, "-c", "import torch; print(torch.__version__); print(torch.version.cuda)"],
        capture_output=True, text=True,
    )
    lines = r.stdout.strip().splitlines()
    return (lines[0], lines[1]) if len(lines) >= 2 else (None, None)


TORCH_BUILD_AT_START = installed_torch_build()
print("torch build on disk, recorded for later comparison:", TORCH_BUILD_AT_START)


def _current_versions(names):
    out = {}
    for name in names:
        r = subprocess.run(
            [sys.executable, "-c", f"import importlib.metadata as m; print(m.version('{name}'))"],
            capture_output=True, text=True,
        )
        out[name] = r.stdout.strip() if r.returncode == 0 else None
    return out


def dry_run_would_change_watched(args):
    """Simulate the install with `pip install --dry-run --report <file>` -- installs
    nothing -- and check whether the packages it WOULD install/upgrade include any of
    WATCHED_PACKAGES at a version different from what's currently on disk. If anything
    would change, the real install never runs at all."""
    report_path = os.path.join(OUTPUT_DIR, "_pip_dry_run_report.json")
    if os.path.exists(report_path):
        os.remove(report_path)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--dry-run", "--quiet",
         "--report", report_path] + args,
        capture_output=True, text=True,
    )
    if result.returncode != 0 or not os.path.exists(report_path):
        return True, {"error": (result.stderr or "no dry-run report produced").strip()[:500]}

    with open(report_path, encoding="utf-8") as f:
        report = _json.load(f)

    current = _current_versions(WATCHED_PACKAGES)
    changes = {}
    for item in report.get("install", []):
        meta = item.get("metadata", {})
        pkg_name = meta.get("name", "")
        pkg_version = meta.get("version")
        for watched in WATCHED_PACKAGES:
            if pkg_name.lower() == watched.lower() and pkg_version != current.get(watched):
                changes[watched] = {"current": current.get(watched), "would_become": pkg_version}
    return (len(changes) > 0), changes


def safe_pip_install(args, label):
    """Install `args`, but only after a dry run confirms it wouldn't touch any
    WATCHED_PACKAGES. Returns True if installed (or already satisfied), False if skipped."""
    would_change, details = dry_run_would_change_watched(args)
    if would_change:
        print(f"{label}: SKIPPING install -- a dry run shows it would change a protected "
              f"package: {details}. Nothing was installed.")
        return False

    before = installed_torch_build()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args, check=True)
    after = installed_torch_build()
    if after != before:
        raise RuntimeError(
            f"{label}: installing it changed the on-disk torch build from {before} to "
            f"{after} even though the dry-run check passed. Stopping the notebook rather "
            f"than continue on a possibly broken environment."
        )
    return True


In [ ]:
# ---- Config ----
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single T4 on purpose

OUTPUT_DIR = OUT_DIR = "/kaggle/working"

BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER_REPO_ID = "FARHANFAST/text2sql-qlora-run-a"
ADAPTER_PATH_IN_REPO = "final_adapter"
assert "CHANGE_ME" not in ADAPTER_REPO_ID

# Unsloth records its own pre-quantized repo name in adapter_config.json; same model.
ACCEPTED_ADAPTER_BASE_NAMES = [
    "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
]

# "dev" for all development. "test" only for the one final run of the chosen variant.
SPLIT = "dev"
assert SPLIT in ("dev", "test"), f"SPLIT must be 'dev' or 'test', got {SPLIT!r}"

# name -> (feedback mode, retry on empty result too?)
VARIANTS = {
    "plain": ("plain", False),
    "error": ("error", False),
    "plain_empty": ("plain", True),
    "error_empty": ("error", True),
}

MAX_RETRIES = 2
SQL_TIMEOUT_SEC = 5.0

SEED = 0
MAX_NEW_TOKENS = 256
MAX_MODEL_LEN = 4096      # matches the Step 7 training cutoff
CHUNK_SIZE = 32            # examples per flush/resume checkpoint
INITIAL_BATCH_SIZE = 8     # halves automatically and retries on CUDA OOM


In [ ]:
# Kaggle's preinstalled torchao (0.10.0) conflicts with the peft version this notebook
# needs -- confirmed on the real Run A evaluation run. Uninstall it up front, before the
# peft preflight cell below. A REMOVAL, not an install pip could resolve into touching
# torch/torchvision/torchaudio, so it doesn't go through the dry-run safety check -- but
# the on-disk torch build is still checked unchanged afterward, same as safe_pip_install.
import importlib.metadata as _im

try:
    _torchao_version = _im.version("torchao")
    _before_torchao_uninstall = installed_torch_build()
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    _after_torchao_uninstall = installed_torch_build()
    if _after_torchao_uninstall != _before_torchao_uninstall:
        raise RuntimeError(
            f"uninstalling torchao changed the on-disk torch build from "
            f"{_before_torchao_uninstall} to {_after_torchao_uninstall} -- stopping rather "
            f"than continue on a possibly broken environment."
        )
    print(f"uninstalled torchao {_torchao_version} (was conflicting with the peft version "
          f"this notebook needs)")
except _im.PackageNotFoundError:
    print("torchao not installed -- nothing to do")


In [ ]:
# peft preflight: make sure it imports cleanly and print its version before any adapter
# is attached below. Only installs it if actually missing, and only after confirming (dry
# run) it wouldn't touch a watched package.
try:
    import peft
    print("peft already installed:", peft.__version__)
except ImportError:
    ok = safe_pip_install(["peft"], "peft")
    if not ok:
        raise RuntimeError("peft is required to load the adapters and its install was "
                            "blocked by the dry-run safety check -- see the message above.")
    import peft
    print("peft installed:", peft.__version__)


In [ ]:
# Kaggle secrets
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
except Exception:
    raise RuntimeError("Missing Kaggle secret HF_TOKEN. Add-ons -> Secrets -> add HF_TOKEN "
                        "(read access is enough), then Run All again.")
print("secret loaded OK")


In [ ]:
# Find the prompts file and every database under /kaggle/input (recursive search,
# never a fixed dataset slug).
import glob
import json


def find_input_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    return matches[0] if matches else None


PROMPTS_FNAME = f"prompts_v2_{SPLIT}.jsonl"
PROMPTS_PATH = find_input_file(PROMPTS_FNAME)
if PROMPTS_PATH is None:
    raise FileNotFoundError(
        f"Missing {PROMPTS_FNAME} under /kaggle/input. Attach the prompts dataset "
        f"(or re-upload data/processed/kaggle_upload/{PROMPTS_FNAME})."
    )
print(f"found {PROMPTS_FNAME} -> {PROMPTS_PATH}")

with open(PROMPTS_PATH, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]
print(f"{len(rows)} questions")

DB_PATHS = {db_id: find_input_file(f"{db_id}.sqlite") for db_id in sorted({r["db_id"] for r in rows})}
missing_dbs = [db_id for db_id, path in DB_PATHS.items() if path is None]
if missing_dbs:
    raise FileNotFoundError(
        f"Missing {len(missing_dbs)} database file(s) under /kaggle/input: "
        + ", ".join(f"{d}.sqlite" for d in missing_dbs)
        + f". Attach the dataset built by scripts/prepare_kaggle_databases.py --split {SPLIT}."
    )
print(f"found all {len(DB_PATHS)} databases")


In [ ]:
%%writefile postprocess.py
"""Clean raw model output down to the SQL string."""
import re


def strip_sql_fences(text: str) -> str:
    """Strip a markdown code fence (```sql ... ``` or ``` ... ```) wrapping a model's raw output."""
    text = text.strip()
    fence = re.match(r"^```(?:sql)?\s*(.*?)\s*```$", text, re.DOTALL | re.IGNORECASE)
    if fence:
        text = fence.group(1).strip()
    text = re.sub(r"^```(?:sql)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()


In [ ]:
%%writefile execute.py
"""Run a single SQL query against a SQLite database, read-only, with a timeout."""
import sqlite3
import threading
from pathlib import Path


def run_query(db_path: Path | str, query: str, timeout: float = 5.0) -> tuple[list | None, str | None]:
    """Execute `query` read-only against db_path.

    Returns (rows, None) on success, or (None, error_message) on failure (bad SQL,
    a write attempt against the read-only connection, or a timeout).

    The timeout uses a background Timer that calls conn.interrupt() instead of
    SIGALRM, since SIGALRM doesn't exist on Windows.
    """
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    except sqlite3.Error as e:
        return None, f"connect error: {e}"

    # matches the official test-suite-sql-eval evaluator's handling of databases
    # with invalid UTF-8 bytes (e.g. a corrupted row in wta_1.sqlite)
    conn.text_factory = lambda b: b.decode(errors="ignore")

    timer = threading.Timer(timeout, conn.interrupt)
    timer.start()
    try:
        rows = conn.execute(query).fetchall()
        return rows, None
    except sqlite3.Error as e:
        return None, str(e)
    finally:
        timer.cancel()
        conn.close()


In [ ]:
%%writefile self_correct.py
"""Self-correction loop: generate SQL, execute it, and retry when it fails.

The loop only uses signals that exist at inference time: the SQLite error message,
or the fact that the query returned no rows. It never sees gold SQL or gold results.

It doesn't know how the SQL is generated. The caller passes `generate_fn`, so the same
loop runs on a GPU with batched generate or on a CPU with llama.cpp.
"""
import time

from execute import run_query
from postprocess import strip_sql_fences

MAX_RETRIES = 2

# feedback modes: what the model is told after a failed attempt
#   "plain": only that the query was wrong
#   "error": the SQLite error message (or that the result was empty)
MODES = ("plain", "error")

PLAIN_FEEDBACK = "That query is not right. Write a corrected SQLite query. Reply with only the SQL query."
ERROR_FEEDBACK = (
    "That query failed with this error: {error}\n"
    "Write a corrected SQLite query. Reply with only the SQL query."
)
EMPTY_FEEDBACK = (
    "That query ran but returned no rows. If the question should have an answer, "
    "write a corrected SQLite query. Reply with only the SQL query."
)


def error_type(error: str) -> str:
    """Bucket a SQLite error message into one of four simple types."""
    e = error.lower()
    if "no such column" in e:
        return "wrong column"
    if "no such table" in e:
        return "wrong table"
    if "syntax error" in e or "incomplete input" in e or "unrecognized token" in e:
        return "syntax"
    return "other"


def execute_attempt(db_path, sql: str, timeout: float = 5.0) -> dict:
    """Run one attempt. status is "ok" (rows came back), "empty" (ran, no rows) or "error"."""
    start = time.time()
    rows, error = run_query(db_path, sql, timeout)
    exec_time = time.time() - start
    if error is not None:
        return {"status": "error", "error": error, "exec_time_sec": exec_time}
    return {"status": "ok" if rows else "empty", "error": None, "exec_time_sec": exec_time}


def feedback_for(attempt: dict, mode: str) -> str:
    if mode == "plain":
        return PLAIN_FEEDBACK
    if attempt["status"] == "error":
        return ERROR_FEEDBACK.format(error=attempt["error"])
    return EMPTY_FEEDBACK


def build_retry_prompt(prompt: str, failed_output: str, feedback: str) -> str:
    """Extend a ChatML prompt with the failed answer and a feedback turn.

    `prompt` ends with "<|im_start|>assistant\\n", so the failed output goes in as that
    assistant turn, followed by a new user turn. This is the same text Qwen's chat
    template produces for [..., assistant, user] with add_generation_prompt=True.
    """
    return (
        f"{prompt}{failed_output}<|im_end|>\n"
        f"<|im_start|>user\n{feedback}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )


def needs_retry(attempt: dict, retry_on_empty: bool) -> bool:
    return attempt["status"] == "error" or (retry_on_empty and attempt["status"] == "empty")


def pick_final(attempts: list[dict]) -> int:
    """Index of the attempt to return: the latest one that executed without an error.

    A retry never replaces a query that ran with one that crashes. If every attempt
    errored, return the last one.
    """
    for i in range(len(attempts) - 1, -1, -1):
        if attempts[i]["status"] != "error":
            return i
    return len(attempts) - 1


def run_self_correction(rows, first_outputs, generate_fn, db_path_fn, mode,
                        retry_on_empty=False, max_retries=MAX_RETRIES, execute_fn=execute_attempt):
    """Run the retry loop for a list of questions, one retry round at a time.

    rows:          [{"idx", "db_id", "question", "prompt"}], prompt already chat-templated
    first_outputs: [{"raw_output", "gen_time_sec"}], the first attempt for each row
    generate_fn:   list of prompts -> list of {"raw_output", "gen_time_sec"}
    db_path_fn:    db_id -> path of its sqlite file

    Working by round (not by question) lets a GPU generate all retries of a round in
    one batch. Returns one record per row with every attempt and the final SQL.
    """
    assert mode in MODES, f"unknown mode: {mode}"

    def make_attempt(row, output, feedback):
        sql = strip_sql_fences(output["raw_output"])
        attempt = {"raw_output": output["raw_output"], "sql": sql,
                   "gen_time_sec": output["gen_time_sec"], "feedback_given": feedback}
        attempt.update(execute_fn(db_path_fn(row["db_id"]), sql))
        return attempt

    attempts = [[make_attempt(row, out, None)] for row, out in zip(rows, first_outputs)]
    prompts = [row["prompt"] for row in rows]

    for _ in range(max_retries):
        todo = [i for i, a in enumerate(attempts) if needs_retry(a[-1], retry_on_empty)]
        if not todo:
            break
        feedbacks = [feedback_for(attempts[i][-1], mode) for i in todo]
        for i, feedback in zip(todo, feedbacks):
            prompts[i] = build_retry_prompt(prompts[i], attempts[i][-1]["raw_output"], feedback)
        outputs = generate_fn([prompts[i] for i in todo])
        for i, output, feedback in zip(todo, outputs, feedbacks):
            attempts[i].append(make_attempt(rows[i], output, feedback))

    records = []
    for row, row_attempts in zip(rows, attempts):
        final = pick_final(row_attempts)
        records.append({
            "idx": row["idx"], "db_id": row["db_id"], "question": row["question"],
            # raw_output / gen_time_sec keep the same meaning as in the plain predictions
            # files, so scripts/evaluate_experiment.py reads this file unchanged
            "raw_output": row_attempts[final]["raw_output"],
            "gen_time_sec": sum(a["gen_time_sec"] for a in row_attempts),
            "exec_time_sec": sum(a["exec_time_sec"] for a in row_attempts),
            "n_attempts": len(row_attempts), "final_attempt": final,
            "attempts": row_attempts,
        })
    return records


In [ ]:
# The three files above are copies of src/postprocess.py, src/execute.py and
# src/self_correct.py from the repo (a test in the repo checks they stay identical).
import sqlite3

import self_correct
from postprocess import strip_sql_fences

# every database must open and answer a trivial query before any GPU time is spent
for _db_id, _path in DB_PATHS.items():
    _attempt = self_correct.execute_attempt(_path, "SELECT count(*) FROM sqlite_master")
    if _attempt["status"] != "ok":
        raise RuntimeError(f"database {_db_id} at {_path} is not readable: {_attempt['error']}")
print(f"all {len(DB_PATHS)} databases open and answer a query | sqlite {sqlite3.sqlite_version}")


In [ ]:
# Tokenizer loaded ONCE -- shared across every adapter below. It doesn't change between
# adapters (only the LoRA weights merged into the base model do).
from transformers import AutoTokenizer

torch.manual_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # required for correct batched causal-LM generation

# Qwen's chat template ends a turn with <|im_end|>, which may not be the tokenizer's
# default eos_token_id -- stop generation at both, same as the Step 4+5 baseline.
eos_ids = {tokenizer.eos_token_id}
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
if im_end_id is not None and im_end_id != getattr(tokenizer, "unk_token_id", None):
    eos_ids.add(im_end_id)
eos_ids = sorted(eos_ids)
print("stopping generation at token ids:", eos_ids, [tokenizer.decode([i]) for i in eos_ids])


In [ ]:
# build_retry_prompt extends the prompt by string concatenation. Check on a toy
# conversation that this gives exactly what the tokenizer's chat template produces.
_msgs = [{"role": "system", "content": "s"}, {"role": "user", "content": "q"}]
_first = tokenizer.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True)
_expected = tokenizer.apply_chat_template(
    _msgs + [{"role": "assistant", "content": "SELECT 1"}, {"role": "user", "content": "fb"}],
    tokenize=False, add_generation_prompt=True,
)
if self_correct.build_retry_prompt(_first, "SELECT 1", "fb") != _expected:
    raise RuntimeError("build_retry_prompt does not match the tokenizer's chat template")
print("retry prompt format matches the chat template")


In [ ]:
def generate_batch_safe(prompts, batch_size):
    """Generate for `prompts` against the CURRENT global `model`, halving batch_size and
    retrying on a CUDA OOM instead of crashing."""
    try:
        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                        max_length=MAX_MODEL_LEN).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                  eos_token_id=eos_ids, pad_token_id=tokenizer.pad_token_id)
        gen_only = out[:, enc["input_ids"].shape[1]:]
        return tokenizer.batch_decode(gen_only, skip_special_tokens=True)
    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise
        torch.cuda.empty_cache()
        if batch_size <= 1:
            raise
        half = batch_size // 2
        print(f"  CUDA OOM at batch_size={batch_size}, halving to {half} and retrying")
        results = []
        for i in range(0, len(prompts), half):
            results.extend(generate_batch_safe(prompts[i:i + half], half))
        return results


def generate_chunk(rows, batch_size):
    """rows: list of {idx, db_id, question, prompt}. Returns raw output strings in the
    SAME order as `rows` -- sorts by prompt length internally, restores original order."""
    order = sorted(range(len(rows)), key=lambda i: len(rows[i]["prompt"]))
    sorted_prompts = [rows[i]["prompt"] for i in order]
    sorted_outputs = []
    for i in range(0, len(sorted_prompts), batch_size):
        batch = sorted_prompts[i:i + batch_size]
        sorted_outputs.extend(generate_batch_safe(batch, batch_size))
    outputs = [None] * len(rows)
    for pos, orig_idx in enumerate(order):
        outputs[orig_idx] = sorted_outputs[pos]
    return outputs


In [ ]:
import shutil
import time

from huggingface_hub import HfApi, snapshot_download
from peft import PeftModel
from transformers import AutoModelForCausalLM

model = None
adapter_config = None


def ensure_model_loaded():
    """Download the adapter, check its base model, merge it into the fp16 base.
    Called right before the first generation, so a fully-cached rerun never loads it."""
    global model, adapter_config
    if model is not None:
        return
    info = HfApi(token=HF_TOKEN).repo_info(repo_id=ADAPTER_REPO_ID)
    print(f"adapter repo {ADAPTER_REPO_ID} visibility: {'private' if info.private else 'PUBLIC'}")
    local_dir = os.path.join(OUTPUT_DIR, "adapters", "run_a")
    snapshot_download(repo_id=ADAPTER_REPO_ID, local_dir=local_dir,
                      allow_patterns=f"{ADAPTER_PATH_IN_REPO}/*", token=HF_TOKEN)
    adapter_dir = os.path.join(local_dir, ADAPTER_PATH_IN_REPO)

    with open(os.path.join(adapter_dir, "adapter_config.json"), encoding="utf-8") as f:
        adapter_config = json.load(f)
    adapter_base = adapter_config.get("base_model_name_or_path")
    if adapter_base not in ACCEPTED_ADAPTER_BASE_NAMES:
        raise ValueError(
            f"adapter at {ADAPTER_REPO_ID}/{ADAPTER_PATH_IN_REPO} was trained against "
            f"'{adapter_base}', which isn't in ACCEPTED_ADAPTER_BASE_NAMES -- refusing to "
            f"attach it to the wrong base."
        )
    print(f"adapter base '{adapter_base}' accepted | r={adapter_config.get('r')}, "
          f"alpha={adapter_config.get('lora_alpha')}")

    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=torch.float16).to("cuda")
    model = PeftModel.from_pretrained(base_model, adapter_dir).merge_and_unload()
    model.eval()
    print("base model + adapter loaded and merged")


def load_jsonl(path):
    out = []
    if not os.path.exists(path):
        return out
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                out.append(json.loads(line))
            except json.JSONDecodeError:
                continue  # a half-written last line from an interrupted run
    return out


def restore_from_previous_run(filename):
    """Copy a file from an attached Notebook Output Files input, if we don't have it yet."""
    path = os.path.join(OUT_DIR, filename)
    if not os.path.exists(path):
        prev = find_input_file(filename)
        if prev is not None:
            shutil.copy(prev, path)
            print(f"found a previous {filename} -> copied {prev}")
    return path


In [ ]:
# ---- First attempt for every question (the no-retry baseline) ----
attempt1_path = restore_from_previous_run("predictions_attempt1.jsonl")
done = {r["idx"]: r for r in load_jsonl(attempt1_path)}
remaining = [r for r in rows if r["idx"] not in done]

if not remaining:
    print(f"attempt 1: already complete ({len(done)}/{len(rows)}) -- skipping")
else:
    print(f"attempt 1: {'RESUMING' if done else 'STARTING FRESH'} -- "
          f"{len(done)}/{len(rows)} done, {len(remaining)} remaining")
    ensure_model_loaded()
    # rewrite the file first so a half-written last line can't survive
    with open(attempt1_path, "w", encoding="utf-8") as f:
        for idx in sorted(done):
            f.write(json.dumps(done[idx]) + "\n")
    with open(attempt1_path, "a", encoding="utf-8") as f:
        for i in range(0, len(remaining), CHUNK_SIZE):
            chunk = remaining[i:i + CHUNK_SIZE]
            t0 = time.time()
            raw_outputs = generate_chunk(chunk, INITIAL_BATCH_SIZE)
            elapsed = time.time() - t0
            for r, raw in zip(chunk, raw_outputs):
                rec = {"idx": r["idx"], "db_id": r["db_id"], "question": r["question"],
                       "raw_output": raw, "gen_time_sec": elapsed / len(chunk)}
                done[r["idx"]] = rec
                f.write(json.dumps(rec) + "\n")
            f.flush()
            os.fsync(f.fileno())
            print(f"attempt 1: {len(done)}/{len(rows)} done, last chunk {elapsed:.1f}s")

first_outputs = [done[r["idx"]] for r in rows]


In [ ]:
# ---- Retry rounds, one variant at a time ----
# Every retry generation is cached on disk by its full prompt. Two variants that build
# the same retry prompt share one generation, and a rerun reuses everything already done.
cache_path = restore_from_previous_run("retry_generation_cache.jsonl")
gen_cache = {r["prompt"]: r for r in load_jsonl(cache_path)}
print(f"retry cache: {len(gen_cache)} generations already on disk")
newly_generated = 0


def generate_fn(prompts):
    global newly_generated
    todo = [p for p in dict.fromkeys(prompts) if p not in gen_cache]
    if todo:
        ensure_model_loaded()
    with open(cache_path, "a", encoding="utf-8") as f:
        for i in range(0, len(todo), CHUNK_SIZE):
            chunk = [{"prompt": p} for p in todo[i:i + CHUNK_SIZE]]
            t0 = time.time()
            raw_outputs = generate_chunk(chunk, INITIAL_BATCH_SIZE)
            elapsed = time.time() - t0
            for c, raw in zip(chunk, raw_outputs):
                rec = {"prompt": c["prompt"], "raw_output": raw, "gen_time_sec": elapsed / len(chunk)}
                gen_cache[c["prompt"]] = rec
                f.write(json.dumps(rec) + "\n")
            f.flush()
            os.fsync(f.fileno())
            newly_generated += len(chunk)
    return [gen_cache[p] for p in prompts]


# the same SQL on the same database is executed once and reused across variants
exec_cache = {}


def execute_fn(db_path, sql):
    key = (db_path, sql)
    if key not in exec_cache:
        exec_cache[key] = self_correct.execute_attempt(db_path, sql, SQL_TIMEOUT_SEC)
    return dict(exec_cache[key])


variant_summaries = {}
for name, (mode, retry_on_empty) in VARIANTS.items():
    records = self_correct.run_self_correction(
        rows, first_outputs, generate_fn, lambda db_id: DB_PATHS[db_id],
        mode=mode, retry_on_empty=retry_on_empty, max_retries=MAX_RETRIES, execute_fn=execute_fn,
    )
    out_path = os.path.join(OUT_DIR, f"selfcorrect_{name}.jsonl")
    with open(out_path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec) + "\n")
    retried = [r for r in records if r["n_attempts"] > 1]
    variant_summaries[name] = {
        "mode": mode, "retry_on_empty": retry_on_empty,
        "n_questions": len(records), "n_retried": len(retried),
        "avg_attempts": sum(r["n_attempts"] for r in records) / len(records),
        "retry_gen_time_sec": sum(a["gen_time_sec"] for r in retried for a in r["attempts"][1:]),
        "first_attempt_gen_time_sec": sum(r["attempts"][0]["gen_time_sec"] for r in records),
        "exec_time_sec": sum(r["exec_time_sec"] for r in records),
        "final_status_counts": {
            s: sum(r["attempts"][r["final_attempt"]]["status"] == s for r in records)
            for s in ("ok", "empty", "error")
        },
    }
    print(f"{name}: wrote {out_path} | {json.dumps(variant_summaries[name])}")

first_status = [self_correct.execute_attempt(DB_PATHS[r["db_id"]], strip_sql_fences(o["raw_output"]),
                                             SQL_TIMEOUT_SEC)["status"] for r, o in zip(rows, first_outputs)]
print("attempt 1 status counts:", {s: first_status.count(s) for s in ("ok", "empty", "error")})
print(f"retry generations this session: {newly_generated} new, {len(gen_cache) - newly_generated} from cache")


In [ ]:
import peft

run_metadata = {
    "base_model_evaluated_on": BASE_MODEL_NAME,
    "adapter_repo": ADAPTER_REPO_ID, "adapter_path_in_repo": ADAPTER_PATH_IN_REPO,
    "adapter_config": adapter_config,  # None if nothing had to be generated this session
    "split": SPLIT, "prompt_version": "v2", "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS, "decoding": "greedy", "eos_token_ids": eos_ids,
    "max_retries": MAX_RETRIES, "sql_timeout_sec": SQL_TIMEOUT_SEC,
    "feedback_templates": {"plain": self_correct.PLAIN_FEEDBACK, "error": self_correct.ERROR_FEEDBACK,
                           "empty": self_correct.EMPTY_FEEDBACK},
    "python_version": sys.version, "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda, "transformers_version": transformers.__version__,
    "peft_version": peft.__version__, "sqlite_version": sqlite3.sqlite_version,
    "gpu_name": torch.cuda.get_device_name(0),
    "attempt1_status_counts": {s: first_status.count(s) for s in ("ok", "empty", "error")},
    "retry_generations_new_this_session": newly_generated,
    "retry_generations_total": len(gen_cache),
    "variants": variant_summaries,
}
with open(os.path.join(OUT_DIR, "run_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)
print("saved run_metadata.json")


## Download from /kaggle/working/ when done
- `predictions_attempt1.jsonl`
- `selfcorrect_plain.jsonl`, `selfcorrect_error.jsonl`, `selfcorrect_plain_empty.jsonl`,
  `selfcorrect_error_empty.jsonl`
- `retry_generation_cache.jsonl`
- `run_metadata.json`

Put them all in `results/step9/` in the repo, then run:

```
python scripts/analyze_self_correction.py
```
